# **任务13 GNN节点嵌入 | GNN - Node embedding**

在先前的图神经网络任务中，我们使用了具体的一些特征值作为图节点的特征向量，对于离散的节点信息，比如每个节点对应一个已知的类别，而我们不能将其拆解为属性，我们可以用先前在 **任务10** 中提到的嵌入层来赋予它们特征向量，同样需要在数据准备时将输入更改为整型张量。

这里就不使用具体的任务来展示了，仅定义模型和模拟前向传播过程。

## 1. 模型定义

In [32]:
import torch
import torch.nn as nn
from torch_geometric.utils import scatter  # 用于批次图池化

class Model(nn.Module):
    def __init__(self, num_nodes, embedding_dim, output_dim, hidden_dim):
        super(Model, self).__init__()
        # 嵌入层，将节点映射到嵌入空间
        self.embedding = nn.Embedding(num_nodes, embedding_dim)
        self.conv1 = GCNConv(embedding_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

        self.relu = nn.ReLU()

    def forward(self, data):
        x, edge_index, batch_idx = data.x, data.edge_index, data.batch
        device = next(self.parameters()).device
        x, edge_index, batch_idx = x.to(device), edge_index.to(device), batch_idx.to(device)

        # 使用嵌入层将节点索引映射为节点特征
        x = self.embedding(x)
        x = self.conv1(x, edge_index)
        x = self.relu(x)
        x = self.conv2(x, edge_index)
        x = self.relu(x)

        x = scatter(x, batch_idx, dim=0, reduce='mean')  # 关键：区分不同图的节点
        # 边选择
        edge_score = self.fc(x)
        return edge_score

## 2. 前向传播

嵌入层的输入需要为整数张量 `torch.zeros()` 方法生成的张量已经是整型了，无需额外处理。

In [33]:
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

batch_size = 8
num_nodes = 5
output_dim = 1
embedding_dim = 16
hidden_dim = 32

input_tensor = torch.arange(num_nodes, dtype=torch.long)

edge_index = torch.tensor([
    [0, 1, 2, 3, 4, 0],
    [4, 3, 3, 1, 0, 2],
], dtype=torch.long)

model = Model(num_nodes=5, embedding_dim=16, output_dim=1, hidden_dim=16)

data = Data(
    x=input_tensor,
    edge_index=edge_index,
)
loader = DataLoader([data for _ in range(batch_size)], batch_size=batch_size)
for batch in loader:
    output_tensor = model(batch)

    print('输入张量：', batch.x.shape)
    print('边索引张量：', batch.edge_index.shape)
    print('输出张量：', output_tensor.shape)

输入张量： torch.Size([40])
边索引张量： torch.Size([2, 48])
输出张量： torch.Size([8, 1])


**任务12**提到过，批次维度依然是 `batch * num_nodes = 8 * 5 = 40`，虽然看起来被混合了但不同图之间不会互相影响。边索引也是同理。